## Preparando os dados via API

In [0]:
!pip install ccxt

In [0]:
import ccxt
import time

par = 'BTC/USDT'
limit = 1000
timeframe = '1h' # 15m / 4h / 1h

# conexão com a API. enableRateLimit para não exceder o limite de requisições
# utilizando Binance US pois o outro está bloqueando meu IP. talvez porque o servidor do databricks fique nos EUA
binance = ccxt.binanceus({'enableRateLimit': True})
print(binance)

# coleta
ohlcv = []
since = binance.parse8601('2025-01-01T00:00:00Z') 
target_total = 50000 # qtd de candles

while len(ohlcv) < target_total:
    # lote de 1000
    new_candles = binance.fetch_ohlcv(par, timeframe=timeframe, since=since, limit=limit)
    
    if not new_candles:
        break
        
    ohlcv += new_candles
    # atualiza o 'since' para o próximo candle após o último recebido
    since = new_candles[-1][0] + 1 
    
    # pausa para não ser banido (rate limit)
    time.sleep(binance.rateLimit / 1000) 

print(f"Total de candles coletados: {len(ohlcv)}")

In [0]:
((len(ohlcv)*60) / 60) / 24

**ccxt:** CryptoCurrency eXchange Trading - conectar e negociar em diversas exchanges de criptomoedas

**Alguns significados**
- O (Open / Abertura): O preço do ativo no exato momento em que o intervalo de tempo começou (ex: o preço às 10:00:00)
- H (High / Máxima): O preço mais alto que o ativo atingiu durante aquele intervalo
- L (Low / Mínima): O preço mais baixo que o ativo atingiu durante aquele intervalo
- C (Close / Fechamento): O preço do ativo no momento em que o intervalo terminou (ex: o preço às 10:59:59)
- V (Volume): A quantidade total do ativo que foi negociada (comprada/vendida) naquele período

In [0]:
import pyspark.sql.types as t
from pyspark.sql.window import Window
import pyspark.sql.functions as f

# criando dataframe
schema = t.StructType([
    t.StructField("timestamp"   ,t.LongType()     ,True)
    ,t.StructField("open"       ,t.DoubleType()   ,True)
    ,t.StructField("high"       ,t.DoubleType()   ,True)
    ,t.StructField("low"        ,t.DoubleType()   ,True)
    ,t.StructField("close"      ,t.DoubleType()   ,True)
    ,t.StructField("volume"     ,t.DoubleType()   ,True)
])
df_ohlcv = spark.createDataFrame(ohlcv, schema)

# convertendo a informação de data para timestamp
df_ohlcv = df_ohlcv.withColumn("timestamp", (df_ohlcv.timestamp / 1000).cast("timestamp"))

 Não está no nosso fuso horário

In [0]:
from pyspark.sql.functions import from_utc_timestamp
df_ohlcv = df_ohlcv.withColumn("timestamp", from_utc_timestamp("timestamp", "America/Sao_Paulo"))
df_ohlcv = df_ohlcv.orderBy(f.col('timestamp').desc())

In [0]:
close_s     = df_ohlcv.select("close")
high_s      = df_ohlcv.select("high")
low_s       = df_ohlcv.select("low")
volume_s    = df_ohlcv.select("volume")

## Feature engineering

In [0]:
w = Window.orderBy("timestamp")

#### Log return (retorno logarítmico)
$$r_t = \ln(P_t) - \ln(P_{t-1})$$
- Cálculo do retorno de um ativo entre dois períodos usando logaritmos naturais
- Permite somar retornos de diferentes períodos
- adequado para séries temporais

In [0]:
df_ohlcv.display()

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

No dia 26 de março às 14:45, houve um pico de volume bem grande, porém, não mexeu nos preços seguintes. seria um sinal de exaustão?

In [0]:
df_feat = df_ohlcv.withColumn(
    "log_return",
    f.log(f.col("close") / f.lag(f.col("close"), 1).over(w))
)
# df_feat.limit(5).display()

#### RSI (Índice de Força Relativa)
$$RSI = 100 - \left( \frac{100}{1 + RS} \right)$$
Onde o RS (Força Relativa) é:
$$RS = \frac{\text{media dos ganhos}}{\text{media das perdas}}$$

- Mede a velocidade e a mudança dos movimentos de preço
- Indica condições de sobrecompra (>70) ou sobrevenda (<30)
- Utilizado para identificar possíveis reversões de tendência
- Geralmente, utiliza-se um período de 14 candles para esse cálculo

In [0]:
periodo = 14
periodo = periodo - 1

delta = f.col("close") - f.lag(f.col("close"), 1).over(w)

ganho = f.when(delta > 0, delta).otherwise(0)
perda = f.when(delta < 0, -delta).otherwise(0)

avg_ganho = f.avg(ganho).over(Window.orderBy("timestamp").rowsBetween(-13, 0))
avg_perda = f.avg(perda).over(Window.orderBy("timestamp").rowsBetween(-13, 0))

rs = avg_ganho / avg_perda
rsi = f.when(avg_perda == 0, 100).otherwise(100 - (100 / (1 + rs)))

df_feat = df_feat.withColumn("rsi", rsi)
# df_feat.limit(5).display()


### Variação de volume
- mede a alteração na intensidade das negociações entre dois períodos
- valida se um movimento de preço tem relevância ou se é apenas um ruído de baixa liquidez
$$V_{change} = \frac{V_t - V_{t-1}}{V_{t-1}}$$
- Vt: Volume do período atual
- Vt-1: Volume do período anterior

In [0]:
df_feat = df_feat.withColumn(
    "vol_change"
    ,f.try_divide((f.col("volume") - f.lag(f.col("volume"), 1).over(w)), f.lag(f.col("volume"), 1).over(w))
)
# df_feat.limit(5).display()


### Médias móveis, desvio padrão móvel e z-score (janela de 20 períodos)

- **Média móvel:** Suaviza as flutuações de preço, mostrando a tendência geral
$$MA_{20} = \frac{1}{20} \sum_{i=0}^{19} P_{t-i}$$

- **Desvio padrão móvel:** Mede a volatilidade dos preços em uma janela de 20 períodos
$$STD_{20} = \sqrt{\frac{1}{20} \sum_{i=0}^{19} (P_{t-i} - MA_{20})^2}$$

- **Z-score:** Indica o quão distante o preço está da média móvel, em termos de desvio padrão
$$Z_{t} = \frac{P_t - MA_{20}}{STD_{20}}$$

- Pt: Preço do período atual
- MA20: Média móvel dos últimos 20 períodos
- STD20: Desvio padrão dos últimos 20 períodos

In [0]:
periodo = 20
w_movel = Window.orderBy("timestamp").rowsBetween(-periodo + 1, 0)

# Média móvel e Desvio padrão móvel do volume
df_feat = (
    df_feat
    # volume
    .withColumn("media_movel_vol", f.avg("volume").over(w_movel))
    .withColumn("media_movel_close", f.avg("close").over(w_movel))
    # desvio padrão
    .withColumn("std_movel_vol", f.stddev("volume").over(w_movel))
    .withColumn("std_movel_close", f.stddev("close").over(w_movel))
)
# df_feat.limit(5).display()

In [0]:
df_feat = (
    df_feat
    # volume
    .withColumn("z_score_vol", (f.col("volume") - f.col("media_movel_vol")) / f.col("std_movel_vol"))
    # fechamento
    .withColumn("z_score_close", (f.col("close") - f.col("media_movel_close")) / f.col("std_movel_close"))
)
# df_feat.limit(5).display()

In [0]:
# limite de anomalia (Z-Score > 3)
df_feat = df_feat.withColumn("vol_anomalia", f.col("z_score_vol") > 3)

# sinal que combina preço + anomalia de volume
df_feat = df_feat.withColumn(
    "vol_signal",
    f.when(
        (f.col("vol_anomalia")) & (f.col("close") > f.col("open")),
        "Forte compra (baleia)"
    ).when(
        (f.col("vol_anomalia")) & (f.col("close") < f.col("open")),
        "Forte venda (dump)"
    ).otherwise("Normal")
)
# df_feat.limit(5).display()

### Assimetria e Curtose (janela móvel)
- **Assimetria:** Mede a simetria da distribuição dos preços em relação à média. Valores positivos indicam cauda à direita, negativos à esquerda.
- **Curtose:** Mede o grau de "achatamento" da distribuição dos preços. Curtose alta indica mais extremos (caudas pesadas).
$$\text{Assimetria}_t = \text{skewness}(P_{t-19}, ..., P_t)$$
$$\text{Curtose}_t = \text{kurtosis}(P_{t-19}, ..., P_t)$$
- Calculadas sobre a janela móvel de 20 períodos usando a coluna de fechamento (`close`)

In [0]:
# assimetria e curtose
df_feat = df_feat.withColumn("assimetria", f.skewness("close").over(w_movel))
df_feat = df_feat.withColumn("curtosi", f.kurtosis("close").over(w_movel))
# df_feat.limit(5).display()

### Preço Típico
- preço médio ponderado de um candle
- utilizado em indicadores como o VWAP e alguns osciladores
$$\text{Typical Price}_t = \frac{H_t + L_t + C_t}{3}$$
- Ht: Máxima do período
- Lt: Mínima do período
- Ct: Fechamento do período

### VP
- VP: Valor negociado ponderado pelo preço típico
- Calculado como: VP = Typical_Price * volume

In [0]:
df_feat = (
  df_feat
  # Typical price
  .withColumn("typical_price", (f.col("high") + f.col("low") + f.col("close")) / 3)
  # VP
  .withColumn("VP", f.col("Typical_Price") * f.col("volume"))
)
# df_feat.limit(5).display()

### VWAP (Volume Weighted Average Price)
- preço médio ponderado pelo volume negociado em um período
- utilizado para avaliar o preço médio de execução de grandes ordens
$$\text{VWAP}_\text{daily} = \frac{\sum_{i=1}^{N} (\text{Typical Price}_i \times \text{Volume}_i)}{\sum_{i=1}^{N} \text{Volume}_i}$$
- N: número de candles no dia
- Typical Price: preço típico do candle
- Volume: volume negociado no candle

### Distância ao VWAP
- Mede o quanto o preço de fechamento está distante do VWAP diário
$$\text{Dist VWAP}_t = \frac{C_t - \text{VWAP}_\text{daily}}{\text{VWAP}_\text{daily}}$$
- Ct: Fechamento do período
- VWAP_daily: VWAP do dia

In [0]:
w_daily = Window.partitionBy(f.to_date("timestamp")).orderBy("timestamp").rowsBetween(Window.unboundedPreceding, 0)

# Daily VWAP
df_feat =(
  df_feat
  .withColumn("date", f.to_date("timestamp"))
  .withColumn("cum_VP", f.sum("VP").over(w_daily))
  .withColumn("cum_Volume", f.sum("volume").over(w_daily))
  .withColumn("daily_VWAP", f.col("cum_VP") / f.col("cum_Volume"))
)

# Distância ao VWAP
df_feat = df_feat.withColumn("dist_VWAP", (f.col("close") - f.col("daily_VWAP")) / f.col("daily_VWAP"))

# drop columns
df_feat = df_feat.drop("Typical_Price", "VP", "date", "cum_VP", "cum_Volume")
# df_feat.limit(10).display()

In [0]:
df_feat.display()

In [0]:
df_feat.groupBy('vol_signal').count().display()

In [0]:
(
    df_feat
    # .select(
    #     'timestamp'
    #     ,'open'
    #     ,'high'
    #     ,'low'
    #     ,'close'
    #     ,'volume'
    # )
    .write
    .format("delta")
    .mode("overwrite")
    .option('overwriteSchema', 'true')
    .saveAsTable("default.tbtc_features")
)